# Lesson 16 - Dataclasses and Type Hints

**Goal:** Describe record structures and function contracts while retaining runtime validation.

**Level:** intermediate. Complete the preceding chapters first. This notebook defines its own
inputs and runs independently. Run cells top to bottom with **Shift+Enter**. Predict results,
inspect the output, and change one input at a time. No external services or third-party packages
are required beyond your Jupyter setup.

Examples include lightweight executable checks. Practice cells intentionally contain no solutions;
expected results help you verify your own work. Temporary files and databases are isolated and cleaned up.

Where salary examples appear, use the course's simplified exercise rules: DA 48% of basic,
HRA 16% capped at 16,000 monthly, PF 12% of basic plus DA, and annual tax 20% only on annual
gross above 12,00,000, divided by 12 for monthly tax. PF is a separate deduction.
These are learning assumptions, not actual tax or payroll rules.

## 1. Type hints document intent

Hints describe expected types; Python does not enforce them automatically. A static type checker can inspect them separately. Runtime checks are still needed at external input boundaries.

In [1]:
def calculate_da(basic: float, rate: float = 0.48) -> float:
    return basic * rate

print(calculate_da(75_000.0))
print(calculate_da.__annotations__)
assert calculate_da(75_000.0) == 36_000

36000.0
{'basic': <class 'float'>, 'rate': <class 'float'>, 'return': <class 'float'>}


## 2. Dataclasses reduce record boilerplate

@dataclass generates methods such as __init__, __repr__, and equality. field(default_factory=list) gives each instance its own list. Optional values can use a union with None.

In [2]:
from dataclasses import dataclass, field

@dataclass
class Employee:
    employee_id: int
    name: str
    basic_pay: float
    department: str | None = None
    notes: list[str] = field(default_factory=list)

asha = Employee(101, "Asha", 52_500)
arun = Employee(102, "Arun", 75_000)
asha.notes.append("Reviewed")
print(asha)
assert arun.notes == []

Employee(employee_id=101, name='Asha', basic_pay=52500, department=None, notes=['Reviewed'])


## 3. Validate post-initialization

__post_init__ can validate initial values. A frozen dataclass prevents ordinary field reassignment, but it is not deep immutability. Prefer immutable fields in a frozen result record.

In [3]:
import math

@dataclass(frozen=True)
class SalaryInput:
    basic: float
    electricity: float = 1000.0

    def __post_init__(self):
        if not all(math.isfinite(value) and value >= 0
                   for value in (self.basic, self.electricity)):
            raise ValueError("Inputs must be finite and nonnegative")

salary = SalaryInput(75_000, 1500)
print(salary)
try:
    SalaryInput(-1)
except ValueError as error:
    print(error)

SalaryInput(basic=75000, electricity=1500)
Inputs must be finite and nonnegative


## 4. Structured results and serialization

Use a small result record to make fields discoverable. asdict converts a dataclass to a dictionary, but does not turn Decimal or dates into JSON-native values. Keep serialization rules explicit.

In [4]:
from dataclasses import asdict

@dataclass(frozen=True)
class PayResult:
    gross: float
    pf: float
    monthly_tax: float

    @property
    def net(self) -> float:
        return self.gross - self.pf - self.monthly_tax

result = PayResult(124_500, 13_320, 4_900)
print(asdict(result), result.net)
assert result.net == 106_280

{'gross': 124500, 'pf': 13320, 'monthly_tax': 4900} 106280


## Practice lab

Define all inputs in each answer cell, use `practice_` names where helpful, and explain important
decisions in comments. Try a task before looking at its expected results. Add at least one normal,
one boundary, and one invalid-input check where the task has an input contract.
Keep file exercises inside TemporaryDirectory until you deliberately choose a permanent output path.

### Exercise 1. Hint a function

Add input and output hints to the salary function and write a docstring for units and rates.

**Check:** Hints describe monthly inputs and distinguish annual from monthly tax.

In [5]:
# Write your solution and checks here.

### Exercise 2. Independent defaults

Construct two employee dataclasses and add a note to only one.

**Check:** The other notes list remains empty.

In [6]:
# Write your solution and checks here.

### Exercise 3. Reject nonfinite values

Validate negative, NaN, and infinite salary inputs.

**Check:** All are rejected; zero is accepted.

In [7]:
# Write your solution and checks here.

### Exercise 4. Immutable result

Create a frozen result dataclass and try to reassign a field inside a caught exception example.

**Check:** dataclasses.FrozenInstanceError is raised.

In [8]:
# Write your solution and checks here.

### Exercise 5. Encode explicit types

Convert a dataclass containing Decimal and date fields into a JSON-ready record.

**Check:** Money and dates use explicit strings, not an implicit float conversion.

In [9]:
# Write your solution and checks here.

## Chapter project

Define typed Employee, SalaryPolicy, and PayResult records. Calculate results with a pure function and serialize a report with explicit monetary formatting.

In [10]:
# Build your chapter project here. Separate inputs, logic, and display.

## Reflection

1. Which input assumptions does your solution enforce, and which does it document?
2. What happens for empty input, a boundary value, or a malformed record?
3. Which parts can be reused without the notebook?
4. What would make the code clearer to another learner?

Restart the kernel and run all cells after completing your work to catch hidden dependencies.

**Next:** `17_testing_and_logging.ipynb` - Testing, Debugging, and Logging.